
## Assignment 1 – Build Your Own Simple Agent

**Goal:** Implement a functional agent that can take a user query, decide what to do, and execute an action (e.g., search, calculation, summarization).

### Tasks
1. Create a Python or Jupyter file.  
2. Define two tools:
   - `search_tool`: searches web using DuckDuckGo (or any simple API).  
   - `calculator_tool`: evaluates math expressions.  
3. Use an instruction-tuned model (e.g., `mistralai/Mistral-7B-Instruct-v0.2`).  
4. Implement a “Thought → Action → Observation → Answer” reasoning loop.  
5. Example input:


In [1]:
!pip install transformers duckduckgo-search torch

In [2]:
from transformers import pipeline
from duckduckgo_search import DDGS
import math

In [3]:
def search_tool(query):

    with DDGS() as ddgs:

        results = list(
            ddgs.text(
                query,
                max_results=3
            )
        )

    if len(results) == 0:
        return "No results found"

    return results[0]["body"]

In [4]:
def calculator_tool(expression):

    try:
        result = eval(expression)

        return result

    except Exception as e:

        return str(e)

In [ ]:
generator = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

In [ ]:
def agent(query):

    print("User Query:")
    print(query)

    print("\nThought:")

    if any(char.isdigit() for char in query):

        print(
            "This looks like a mathematical problem."
        )

        print("\nAction:")
        print("calculator_tool")

        observation = calculator_tool(
            query
        )

    else:

        print(
            "This requires information retrieval."
        )

        print("\nAction:")
        print("search_tool")

        observation = search_tool(
            query
        )

    print("\nObservation:")
    print(observation)

    answer = f"""
    Based on the observation:

    {observation}
    """

    print("\nAnswer:")
    print(answer)

    return answer

In [ ]:
agent(
    "25 * 12"
)

In [ ]:
agent(
    "What is artificial intelligence?"
)


## Assignment 2 – Add Memory to Your Agent

**Goal:** Extend your agent with memory and contextual continuity.

### Tasks
1. Add a simple key–value memory (`dict`) or use LangChain’s `ConversationBufferMemory`.  
2. Ensure your agent recalls past interactions.


In [ ]:
memory = {}

In [ ]:
def memory_agent(user_id, query):

    print("User:", query)

    if user_id not in memory:
        memory[user_id] = []

    memory[user_id].append(query)

    print("\nMemory:")

    for item in memory[user_id]:
        print("-", item)

    return "Stored in memory."

In [ ]:
memory_agent(
    "student1",
    "My favorite subject is AI"
)

In [ ]:
memory_agent(
    "student1",
    "I want to learn LangChain"
)

In [ ]:
def recall_memory(user_id):

    if user_id not in memory:
        return "No memory found"

    return memory[user_id]

In [ ]:
print(
    recall_memory("student1")
)

Reflection Answer
How does memory improve an agent?

Memory allows the agent to remember previous interactions and maintain context across conversations. Without memory, every user query is treated independently. With memory, the agent can personalize responses, recall past information, and provide more coherent multi-turn conversations.

Difference Between Stateless and Stateful Agents

| Stateless Agent                 | Stateful Agent             |
| ------------------------------- | -------------------------- |
| No memory                       | Maintains memory           |
| Treats each query independently | Uses previous interactions |
| No personalization              | Personalized responses     |
| Simple design                   | More intelligent behavior  |


## Assignment 3 – Multi-Agent Collaboration

**Goal:** Build two or more specialized agents that collaborate.

### Agents
- **SearchAgent:** retrieves factual information  
- **AnalysisAgent:** performs reasoning or computation  
- **ReportAgent:** creates a human-readable summary  

### Orchestration
Implement a simple controller or coordinator (sequential or rule-based).


In [ ]:
!pip install duckduckgo-search

In [ ]:
from duckduckgo_search import DDGS

In [ ]:
class SearchAgent:

    def search(self, query):

        with DDGS() as ddgs:

            results = list(
                ddgs.text(
                    query,
                    max_results=1
                )
            )

        if len(results) == 0:
            return "No information found"

        return results[0]["body"]

In [ ]:
class AnalysisAgent:

    def analyze(self, text):

        word_count = len(
            text.split()
        )

        return (
            f"The retrieved text contains "
            f"{word_count} words."
        )

In [ ]:
class ReportAgent:

    def create_report(
        self,
        query,
        search_result,
        analysis
    ):

        report = f"""
        QUERY:
        {query}

        SEARCH RESULT:
        {search_result}

        ANALYSIS:
        {analysis}
        """

        return report

In [ ]:
class Coordinator:

    def __init__(self):

        self.search_agent = SearchAgent()

        self.analysis_agent = AnalysisAgent()

        self.report_agent = ReportAgent()

    def run(self, query):

        print("Step 1: Search Agent")

        search_result = (
            self.search_agent.search(
                query
            )
        )

        print("Done")

        print("\nStep 2: Analysis Agent")

        analysis = (
            self.analysis_agent.analyze(
                search_result
            )
        )

        print("Done")

        print("\nStep 3: Report Agent")

        report = (
            self.report_agent.create_report(
                query,
                search_result,
                analysis
            )
        )

        return report

In [ ]:
system = Coordinator()

result = system.run(
    "What is artificial intelligence?"
)

print(result)

Reflection Answer
Why use multiple agents instead of one large agent?

Using multiple agents allows specialization. Each agent focuses on a specific task, improving modularity, maintainability, and scalability. It also makes debugging easier because each component has a clear responsibility.

Advantages of Multi-Agent Systems:

Better task separation
Easier maintenance
Improved scalability
Reusable components
More flexible workflows

Challenges:

Communication between agents
Increased complexity
Coordination overhead
Error propagation between agents
Architecture Diagram

                 User Query
                      |
                      V
              +---------------+
              |  Coordinator  |
              +---------------+
                      |
      --------------------------------
      |              |               |
      V              V               V
+------------+ +-------------+ +------------+
| SearchAgent| |AnalysisAgent| |ReportAgent |
+------------+ +-------------+ +------------+
      |              |               |
      --------------------------------
                      |
                      V
               Final Report

## Assignment 4 – Explore Orchestration Frameworks

**Goal:** Implement orchestration with a framework like `LangGraph` or `CrewAI`.

### Tasks
1. Rebuild your multi-agent setup in one of:
   - LangGraph (graph-based orchestration)
   - CrewAI (role-based orchestration)
2. Define node-level roles and interactions.
3. Add a **conditional branch**:
   - e.g., “If topic = finance → use FinanceAgent; else → use GeneralAgent.”


In [ ]:
!pip install crewai

In [ ]:
from crewai import Agent
from crewai import Task
from crewai import Crew

In [ ]:
finance_agent = Agent(

    role="Finance Expert",

    goal="Answer finance related questions",

    backstory="Expert in finance and investments",

    verbose=True
)

In [ ]:
general_agent = Agent(

    role="General Knowledge Expert",

    goal="Answer general questions",

    backstory="Expert in general knowledge",

    verbose=True
)

In [ ]:
report_agent = Agent(

    role="Report Writer",

    goal="Create final report",

    backstory="Produces human readable summaries",

    verbose=True
)

In [ ]:
def select_agent(query):

    finance_keywords = [

        "stock",
        "investment",
        "finance",
        "bank",
        "market"
    ]

    query = query.lower()

    if any(
        word in query
        for word in finance_keywords
    ):

        return finance_agent

    return general_agent

In [ ]:
def create_tasks(query):

    selected_agent = select_agent(
        query
    )

    task1 = Task(

        description=f"""
        Answer this question:

        {query}
        """,

        agent=selected_agent
    )

    task2 = Task(

        description="""
        Create a concise report
        based on the answer.
        """,

        agent=report_agent
    )

    return [task1, task2]

In [ ]:
def run_crew(query):

    tasks = create_tasks(
        query
    )

    crew = Crew(

        agents=[
            finance_agent,
            general_agent,
            report_agent
        ],

        tasks=tasks,

        verbose=True
    )

    result = crew.kickoff()

    return result

In [ ]:
result = run_crew(

    "What are the benefits of stock investments?"
)

print(result)

In [ ]:
result = run_crew(

    "What is artificial intelligence?"
)

print(result)

                     User Query
                          |
                          V
                 +----------------+
                 |   Controller    |
                 +----------------+
                          |
            ---------------------------
            |                         |
            V                         V
    FinanceAgent              GeneralAgent
            |                         |
            ----------+---------------
                     |
                     V
               ReportAgent
                     |
                     V
                Final Answer

Reflection Answers
Why use orchestration frameworks?

Orchestration frameworks manage communication and task flow between multiple agents. They simplify coordination, routing, and execution of complex workflows.

What is the benefit of conditional branching?

Conditional branching allows the system to route tasks to specialized agents based on the query type. This improves accuracy and efficiency because each agent focuses on its area of expertise.

Difference Between Single-Agent and Multi-Agent Systems
| Single Agent                 | Multi-Agent                              |
| ---------------------------- | ---------------------------------------- |
| One model handles everything | Specialized agents handle specific tasks |
| Simpler                      | More scalable                            |
| Harder to optimize           | Easier to extend                         |
| Limited specialization       | Better expertise                         |